# TailorTalk — AI-Powered Visual Saree Search (Production Notebook)

TailorTalk is an AI-powered visual search assistant for a saree catalogue.

### Pipeline Overview
User Image → DINOv2 Visual Feature Extraction → L2 Normalized Vector → FAISS Similarity Search (IndexFlatIP) → Aligned Metadata Lookup → Visually Similar Sarees → LangChain / Gemini Agent → Gradio Frontend Interface

## 1. Environment & Dependencies Installation

In [1]:
!pip install -q faiss-cpu transformers torch torchvision pandas numpy pillow requests langchain langchain-google-genai gradio

import os
import sys
import glob
import requests
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoImageProcessor, AutoModel

print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)
print("FAISS Version:", faiss.__version__)


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Python Version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
PyTorch Version: 2.13.0+cpu
FAISS Version: 1.15.0


## 2. Configuration & System Paths

In [2]:
DATA_DIR = "data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
CSV_PATH = os.path.join(DATA_DIR, "saree.csv")
EMBEDDINGS_PATH = os.path.join(DATA_DIR, "dinov2_embeddings.npy")
METADATA_PATH = os.path.join(DATA_DIR, "searchable_metadata.csv")
FAISS_INDEX_PATH = os.path.join(DATA_DIR, "dinov2_faiss.index")

MODEL_NAME = "facebook/dinov2-small"
EMBEDDING_DIM = 384

os.makedirs(IMAGE_DIR, exist_ok=True)
print(f"Setup complete. Output directories ready at '{DATA_DIR}'.")

Setup complete. Output directories ready at 'data'.


## 3. Load & Inspect Full Product Catalogue

In [3]:
df_full = pd.read_csv(CSV_PATH)
print(f"Full Catalogue Rows: {len(df_full)}")
print(f"Catalogue Columns: {list(df_full.columns)}")
df_full.head(3)

Full Catalogue Rows: 1074
Catalogue Columns: ['Name', 'SKU', 'Stock', 'Retail Price', 'Discounted Price', 'image_url', 'Website Link']


,Name,SKU,Stock,Retail Price,Discounted Price,image_url,Website Link
0,Pashmina - Banarasi Saree - Pink Colour QS204820,QS204820,1,6000,3150,https://byrappasilk.in/storage/uploads/bsrKlEU...,https://byrappasilks.in/shop/pashmina_banarasi...
1,Organza Tissue Sarees - White & Gold Colour QA...,QA255622,0,5495,5020,https://byrappasilk.in/storage/uploads/1cssgxd...,https://byrappasilks.in/shop/organza_tissue_sa...
2,Floral Organza Saree - Red Colour QA254685,QA254685,0,10995,10045,https://byrappasilk.in/storage/uploads/qg47mgX...,https://byrappasilks.in/shop/floral_organza_sa...


## 4. Image Dataset Verification & Download Handling

In [4]:
def download_single_image(args):
    idx, url = args
    target_path = os.path.join(IMAGE_DIR, f"{idx}.jpg")
    if os.path.exists(target_path) and os.path.getsize(target_path) > 0:
        return idx, True
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            img = Image.open(BytesIO(res.content)).convert("RGB")
            img.save(target_path, "JPEG")
            return idx, True
    except Exception:
        pass
    return idx, False

print("Verifying and downloading missing catalogue images...")
download_tasks = [(idx, row["image_url"]) for idx, row in df_full.iterrows()]
with ThreadPoolExecutor(max_workers=10) as executor:
    download_results = list(executor.map(download_single_image, download_tasks))

existing_files = set(os.listdir(IMAGE_DIR))
print(f"Verification Complete. Available local images: {len(existing_files)}")

Verifying and downloading missing catalogue images...
Verification Complete. Available local images: 1069


## 5. DINOv2 Embedding Pipeline Initialization

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading DINOv2 model '{MODEL_NAME}' on device: {device}")

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def extract_dinov2_embedding(pil_image):
    inputs = processor(images=pil_image.convert("RGB"), return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Extract CLS token vector
    vec = outputs.last_hidden_state[:, 0].cpu().numpy().squeeze().astype("float32")
    norm = np.linalg.norm(vec)
    if norm > 1e-12:
        vec = vec / norm
    return vec

print("DINOv2 Feature Extractor initialized successfully.")

Loading DINOv2 model 'facebook/dinov2-small' on device: cpu


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 Feature Extractor initialized successfully.


## 6. Catalogue Embeddings & Synchronous Metadata Generation

In [6]:
valid_indices = []
embeddings_list = []
searchable_records = []

print("Generating embeddings and constructing strict 1-to-1 index mapping...")
for idx, row in df_full.iterrows():
    img_filename = f"{idx}.jpg"
    img_path = os.path.join(IMAGE_DIR, img_filename)
    
    if os.path.exists(img_path):
        try:
            with Image.open(img_path) as img:
                emb = extract_dinov2_embedding(img)
                embeddings_list.append(emb)
                
                rec = row.to_dict()
                rec["catalogue_row_index"] = idx
                rec["image_file"] = img_filename
                searchable_records.append(rec)
                valid_indices.append(idx)
        except Exception as e:
            print(f"Warning: Could not process image {img_filename}: {e}")

embeddings_array = np.vstack(embeddings_list).astype("float32")
searchable_df = pd.DataFrame(searchable_records)

# Cache artifacts
np.save(EMBEDDINGS_PATH, embeddings_array)
searchable_df.to_csv(METADATA_PATH, index=False)

print(f"Embeddings shape: {embeddings_array.shape}")
print(f"Searchable Metadata rows: {len(searchable_df)}")
assert len(embeddings_array) == len(searchable_df), "MISMATCH DETECTED: Vector count != Metadata rows!"

Generating embeddings and constructing strict 1-to-1 index mapping...
Embeddings shape: (1069, 384)
Searchable Metadata rows: 1069


## 7. FAISS Index Construction & Validation

In [7]:
# Construct FAISS Inner Product index (equivalent to Cosine Similarity with normalized vectors)
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings_array)
faiss.write_index(index, FAISS_INDEX_PATH)

print(f"FAISS Index built and saved with {index.ntotal} vectors of dimension {index.d}.")

FAISS Index built and saved with 1069 vectors of dimension 384.


## 8. Visual Search Core Function

In [9]:
def search_similar_sarees(image_input, top_k=5, filter_out_of_stock=False):
    """
    Given an image path or PIL image object, extracts its DINOv2 embedding,
    queries the FAISS index, and returns matching product metadata.
    """
    if isinstance(image_input, str):
        if not os.path.exists(image_input):
            raise FileNotFoundError(f"Input image not found: {image_input}")
        query_img = Image.open(image_input)
    else:
        query_img = image_input
        
    # Generate query embedding using identical DINOv2 pipeline
    query_vec = extract_dinov2_embedding(query_img).reshape(1, -1)
    
    # Search additional entries to account for potential duplicate SKU deduplication
    search_k = top_k * 4
    scores, indices = index.search(query_vec, search_k)
    
    results = []
    seen_skus = set()
    
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(searchable_df):
            continue
            
        row = searchable_df.iloc[idx]
        sku = row["SKU"]
        stock = int(row["Stock"])
        
        if filter_out_of_stock and stock <= 0:
            continue
        if sku in seen_skus:
            continue
            
        seen_skus.add(sku)
        results.append({
            "faiss_index": int(idx),
            "image_file": row["image_file"],
            "name": row["Name"],
            "sku": sku,
            "similarity_score": float(score),
            "retail_price": float(row["Retail Price"]),
            "discounted_price": float(row["Discounted Price"]),
            "stock": stock,
            "website_link": row["Website Link"]
        })
        
        if len(results) == top_k:
            break
            
    return results

## 9. Diagnostic & Mapping Sanity Verification

In [11]:
print("=== RUNNING SANITY TEST 1: Direct Metadata Verification for 10.jpg ===")
row_10 = searchable_df[searchable_df["image_file"] == "10.jpg"].iloc[0]
print(f"File: {row_10['image_file']}")
print(f"Name: {row_10['Name']}")
print(f"SKU: {row_10['SKU']}")
assert row_10["SKU"] == "QA255417", f"Expected QA255417, got {row_10['SKU']}"
print("Sanity Test 1 Passed!\n")

print("=== RUNNING SANITY TEST 2 & 3: Querying FAISS with 10.jpg ===")
test_img_path = os.path.join(IMAGE_DIR, "10.jpg")
test_results = search_similar_sarees(test_img_path, top_k=5)

print(f"Top Result SKU: {test_results[0]['sku']} (Expected: QA255417)")
print(f"Top Result Similarity Score: {test_results[0]['similarity_score']:.4f}")
assert test_results[0]["sku"] == "QA255417", "Visual Search Mapping Failed! Top match does not equal source image."
print("Sanity Tests 2 & 3 Passed!\n")

print("=== DIAGNOSTIC RETRIEVAL TABLE ===")
diag_df = pd.DataFrame(test_results)[["faiss_index", "image_file", "name", "sku", "similarity_score", "stock"]]
display(diag_df)

=== RUNNING SANITY TEST 1: Direct Metadata Verification for 10.jpg ===
File: 10.jpg
Name: Pashmina - Banarasi Saree -Navy Blue Colour QA255417
SKU: QA255417
Sanity Test 1 Passed!

=== RUNNING SANITY TEST 2 & 3: Querying FAISS with 10.jpg ===
Top Result SKU: QA255417 (Expected: QA255417)
Top Result Similarity Score: 1.0000
Sanity Tests 2 & 3 Passed!

=== DIAGNOSTIC RETRIEVAL TABLE ===


,faiss_index,image_file,name,sku,similarity_score,stock
0,10,10.jpg,Pashmina - Banarasi Saree -Navy Blue Colour QA...,QA255417,1.000000,1
1,0,0.jpg,Pashmina - Banarasi Saree - Pink Colour QS204820,QS204820,0.929563,1
2,9,9.jpg,Pashmina - Banarasi Saree -Cream Colour QA255621,QA255621,0.924427,1
3,8,8.jpg,Munga Crape Saree With Cream Colour AA313401,AA313401,0.919586,1
4,12,12.jpg,Satin Printed Sarees - Blue Colour QA255500,QA255500,0.886790,0


## 10. LangChain Agent & Fallback Handler

In [12]:
# ============================================================
# 10. LANGCHAIN TOOL + GEMINI AGENT + FALLBACK
# ============================================================

import os
import json

from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


# ------------------------------------------------------------
# 1. VISUAL SEARCH TOOL
# ------------------------------------------------------------

@tool
def visual_saree_search_tool(image_path: str) -> str:
    """
    Search the saree catalogue for visually similar sarees.

    The image_path must point to a local saree image.
    Product information is retrieved only from the verified
    FAISS + catalogue search pipeline.
    """

    try:
        results = search_similar_sarees(
            image_path,
            top_k=5,
            filter_out_of_stock=False
        )

        return json.dumps(
            results,
            ensure_ascii=False,
            default=str
        )

    except Exception as e:
        return json.dumps({
            "error": str(e)
        })


# ------------------------------------------------------------
# 2. DIRECT / NON-GEMINI FALLBACK
# ------------------------------------------------------------

def run_direct_visual_search(image_path, top_k=5):
    """
    Reliable fallback that directly calls the verified
    DINOv2 + FAISS visual search pipeline.
    """

    results = search_similar_sarees(
        image_path,
        top_k=top_k,
        filter_out_of_stock=False
    )

    output = "### TailorTalk Visual Search Results\n\n"

    for i, result in enumerate(results, 1):

        output += (
            f"### {i}. {result['name']}\n"
        )

        output += (
            f"- **Similarity:** "
            f"{result['similarity_score']:.4f}\n"
        )

        output += (
            f"- **SKU:** "
            f"{result['sku']}\n"
        )

        output += (
            f"- **Retail Price:** "
            f"₹{result['retail_price']:,.2f}\n"
        )

        output += (
            f"- **Discounted Price:** "
            f"₹{result['discounted_price']:,.2f}\n"
        )

        output += (
            f"- **Stock:** "
            f"{result['stock']} units\n"
        )

        output += (
            f"- **Product Link:** "
            f"{result['website_link']}\n\n"
        )

    return output


# ------------------------------------------------------------
# 3. GEMINI AGENT
# ------------------------------------------------------------

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

gemini_llm = None
gemini_agent = None


if GEMINI_API_KEY:

    try:

        gemini_llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=GEMINI_API_KEY,
            temperature=0
        )

        # Use LangChain's tool-calling interface.
        #
        # We keep the agent simple and reliable:
        # Gemini receives the user's request and can call
        # visual_saree_search_tool when visual search is needed.

        gemini_llm_with_tools = gemini_llm.bind_tools(
            [visual_saree_search_tool]
        )

        print(
            "Gemini tool-calling model initialized successfully."
        )

    except Exception as e:

        print(
            "Gemini initialization failed:"
        )

        print(e)

        gemini_llm = None
        gemini_llm_with_tools = None

else:

    gemini_llm_with_tools = None

    print(
        "GEMINI_API_KEY not found."
    )

    print(
        "Direct FAISS fallback will be used."
    )


# ------------------------------------------------------------
# 4. GEMINI RESPONSE FUNCTION
# ------------------------------------------------------------

def run_gemini_visual_search(
    image_path,
    user_query="Find visually similar sarees."
):
    """
    Runs the Gemini tool-calling workflow.

    Gemini can decide to call the visual search tool.
    If Gemini is unavailable or fails, the verified
    direct FAISS search is used instead.
    """

    if gemini_llm_with_tools is None:

        return run_direct_visual_search(
            image_path,
            top_k=5
        )


    try:

        prompt = f"""
You are TailorTalk, an AI saree shopping assistant.

The user wants help finding sarees visually similar
to an uploaded saree image.

User request:
{user_query}

The uploaded image is available at:
{image_path}

You MUST use the visual_saree_search_tool to retrieve
actual catalogue results.

Do NOT invent:
- saree names
- SKUs
- prices
- stock
- product links

Only use product information returned by the tool.

After receiving the tool results, present the best
matching sarees clearly and concisely.
"""

        messages = [
            (
                "system",
                "You are a helpful saree visual-search assistant."
            ),
            (
                "human",
                prompt
            )
        ]

        response = gemini_llm_with_tools.invoke(
            messages
        )


        # ----------------------------------------------------
        # Handle tool calls
        # ----------------------------------------------------

        if getattr(
            response,
            "tool_calls",
            None
        ):

            tool_results = []

            for tool_call in response.tool_calls:

                if tool_call["name"] != (
                    "visual_saree_search_tool"
                ):
                    continue

                tool_result = visual_saree_search_tool.invoke(
                    tool_call["args"]
                )

                tool_results.append(
                    tool_result
                )


            # If a tool was called, ask Gemini to format
            # the actual returned catalogue information.

            if tool_results:

                final_messages = messages + [
                    response
                ]

                for tool_result in tool_results:

                    final_messages.append(
                        (
                            "tool",
                            tool_result
                        )
                    )

                final_response = (
                    gemini_llm.invoke(
                        final_messages
                    )
                )

                return final_response.content


        # ----------------------------------------------------
        # If Gemini answered without calling the tool,
        # use direct search to guarantee actual results.
        # ----------------------------------------------------

        return run_direct_visual_search(
            image_path,
            top_k=5
        )


    except Exception as e:

        print(
            "Gemini search failed."
        )

        print(
            f"Reason: {e}"
        )

        print(
            "Using direct FAISS fallback."
        )

        return run_direct_visual_search(
            image_path,
            top_k=5
        )


# ------------------------------------------------------------
# 5. UNIFIED TAILORTALK PIPELINE
# ------------------------------------------------------------

def run_tailortalk_pipeline(
    image_path,
    user_query=""
):
    """
    Main application entry point.

    Gemini is used when configured.
    Direct DINOv2 + FAISS search is the fallback.
    """

    if not user_query.strip():

        user_query = (
            "Find 5 sarees visually similar "
            "to this saree."
        )


    return run_gemini_visual_search(
        image_path,
        user_query
    )


print(
    "✅ Section 10 ready."
)

print(
    "Visual search tool: READY"
)

if gemini_llm_with_tools is not None:

    print(
        "Gemini tool-calling: READY"
    )

else:

    print(
        "Gemini tool-calling: NOT CONFIGURED"
    )

print(
    "Direct FAISS fallback: READY"
)

Gemini tool-calling model initialized successfully.
✅ Section 10 ready.
Visual search tool: READY
Gemini tool-calling: READY
Direct FAISS fallback: READY


## 11. Gradio Frontend Interface

In [13]:
# ============================================================
# 11. GRADIO FRONTEND
# ============================================================

import os
import gradio as gr


# ------------------------------------------------------------
# GRADIO SEARCH FUNCTION
# ------------------------------------------------------------

def gradio_search(
    image,
    user_query
):
    """
    Handles the Gradio upload and sends the image
    through the TailorTalk visual-search pipeline.
    """

    if image is None:

        return (
            "## Please upload a saree image first."
        )


    temp_path = os.path.join(
        DATA_DIR,
        "temp_query.jpg"
    )


    try:

        # Save uploaded PIL image
        image.convert("RGB").save(
            temp_path,
            "JPEG"
        )


        if not user_query:

            user_query = (
                "Find 5 sarees visually similar "
                "to this saree."
            )


        result = run_tailortalk_pipeline(
            temp_path,
            user_query
        )


        return result


    except Exception as e:

        return (
            "## Search Error\n\n"
            f"`{str(e)}`"
        )


    finally:

        # Clean up temporary query image
        if os.path.exists(temp_path):

            try:
                os.remove(temp_path)

            except Exception:
                pass


# ------------------------------------------------------------
# GRADIO INTERFACE
# ------------------------------------------------------------

demo = gr.Interface(

    fn=gradio_search,

    inputs=[

        gr.Image(
            type="pil",
            label="Upload Saree Image"
        ),

        gr.Textbox(
            label="What are you looking for?",
            placeholder=(
                "e.g. Find 5 sarees similar to this image"
            )
        )
    ],

    outputs=gr.Markdown(
        label="TailorTalk Recommendations"
    ),

    title=(
        "TailorTalk — AI-Powered "
        "Visual Saree Search"
    ),

    description=(
        "Upload a saree image to discover "
        "visually similar sarees from the catalogue."
    ),

    examples=None
)


print(
    "✅ Gradio interface created successfully."
)

print(
    "Run the next line to launch the application:"
)

print(
    "demo.launch(share=True)"
)

✅ Gradio interface created successfully.
Run the next line to launch the application:
demo.launch(share=True)


In [ ]:
# ============================================================
# LAUNCH TAILORTALK
# ============================================================

demo.launch(
    share=True
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2986679c5a4ed35730.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gemini search failed.
Reason: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}
Using direct FAISS fallback.
Gemini search failed.
Reason: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}
Using direct FAISS fallback.
Gemini search failed.
Reason: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'cod

: 